# Reforestación: cuánto comprar y dónde plantar

Este notebook conserva el recorrido didáctico del modelo espacial de **cuotas fijas** (secciones 2–14) y ejecuta el sistema integrado en la sección 15. La primera celda permite elegir `MODO = "fijo"` o `"economico"`, un ID, una lista o `"todos"`.

El modo económico resuelve primero un MILP por polígono y entrega las cuotas a DSATUR y búsqueda local. El recorrido inicial con cuotas fijas sirve como referencia explicativa; sus resultados no deben confundirse con los del modo económico de la sección 15.

Las salidas nuevas se guardan en carpetas únicas; los resultados históricos permanecen intactos. Un escenario es suficiente para comprobar ejecución, pero no para estimar dispersión Monte Carlo. Para el experimento completo configura 12 escenarios.

**Supuestos pendientes:** ±10% sobre las cuotas finales originales; costos únicamente de adquisición; geometría simulada y competencia provisional. Un menor J no demuestra mayor supervivencia.

In [ ]:
# OPCIONES PRINCIPALES — editar aquí antes de ejecutar todo.
MOTOR = "cpp"  # o "cpp"; compilar antes con scripts/compilar_cpp.py
MODO = "economico"  # o "fijo"
POLIGONOS = "todos"              # ID 1–30, lista (p. ej. [5,9]) o "todos"
ESCENARIOS_MC = 1         # Monte Carlo del polígono detallado; usar 2 para probar rápido
ESCENARIOS_LOTE = 1       # escenarios por polígono; 1 para probar, 12 para Monte Carlo
EJECUTAR_COMPARACION_PRESUPUESTOS = False

from pathlib import Path
import sys
from dataclasses import replace, asdict
from time import perf_counter
import inspect
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, Code
PROJECT_ROOT = Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT/'src'))
from reforestacion.config import Config, perfil
from reforestacion.datos import Datos, cargar_competencia
from reforestacion.geometria import dimensiones, flujos, simular_existentes, candidatas, construir_grafo
from reforestacion.algoritmos import objetivo, base_aleatoria, dsatur, busqueda_local
from reforestacion.validacion import validar_solucion
from reforestacion.experimento import ejecutar_lote, guardar_ejecucion, agregar, seleccionar_poligonos, guardar_monte_carlo_por_poligono
from reforestacion import visualizacion as vis
from reforestacion.economia import presupuestos, diagnostico_flexibilidad
from reforestacion.reportes import guardar_modelo_completo, guardar_configuracion

ROOT = PROJECT_ROOT
from datetime import datetime, timezone
RESULTADOS = ROOT/'resultados'/'ejecuciones'/('tutorial_fijo_'+datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ'))
RESULTADOS.mkdir(parents=True, exist_ok=True)
poligonos_ids = seleccionar_poligonos(POLIGONOS)
p = poligonos_ids[0]
OUT = RESULTADOS/f'P{p}'
FIGURAS = OUT/'graficas'
FIGURAS.mkdir(parents=True, exist_ok=True)  # detalle paso a paso y figuras del primer polígono elegido
cfg = perfil('completo', poligonos=tuple(poligonos_ids), escenarios=ESCENARIOS_MC, motor=MOTOR)
from reforestacion.motores import obtener_motor
motor = obtener_motor(MOTOR)
candidatas, dsatur, busqueda_local = motor.candidatas, motor.dsatur, motor.busqueda_local
cfg_lote = replace(cfg, escenarios=ESCENARIOS_LOTE)  # valida también este presupuesto
print('Polígonos seleccionados:', poligonos_ids)
print(f'Detalle y Monte Carlo: P{p}; escenarios MC: {ESCENARIOS_MC}')
if len(poligonos_ids) > 1:
    print(f'Lote seleccionado: {len(poligonos_ids)} polígonos, {ESCENARIOS_LOTE} escenario(s) por polígono.')
plt.rcParams.update({'font.size':10,'figure.dpi':110,'axes.spines.top':False,'axes.spines.right':False})
def mostrar(fig, nombre):
    fig.savefig(FIGURAS/(nombre+'.png'),dpi=140,bbox_inches='tight')
    plt.show()
    plt.close(fig)
display(pd.Series(asdict(cfg),name='Configuración'))

## 2. Datos proporcionados por el socio formador
`data/especies.csv`: especie, género, meta por ha y costo unitario. `data/inventario.csv`: área y conteos por especie en 30 polígonos. `data/supervivencia_porcentajes.csv`: contexto histórico, **sin uso como predicción ni como C**. `data/fuentes_sha256.json` registra huellas de documentos originales; no contiene esos documentos y no se usa para regenerar datos.

Los nombres Opuntia engelmannii y Opuntia streptacantha ya están normalizados en CSV. La moneda no está especificada: u.m. P13=5.05 ha es una interpretación editorial pendiente; las etiquetas del mapa no son necesariamente los IDs tabulados. El cargador rechaza duplicados, conteos no enteros, especies ausentes y áreas inconsistentes.

In [ ]:
datos = Datos(ROOT)
S = datos.S
area, existentes, cuotas, nuevas = datos.poligono(p,cfg)
tabla = datos.especies.copy()
tabla['existentes'] = existentes
tabla['cuota_final'] = cuotas
tabla['nuevas'] = nuevas
tabla['costo_nuevas_um'] = nuevas * tabla.costo_unitario
display(tabla)
print(f'P{p}: {area} ha; {existentes.sum()} existentes + {nuevas.sum()} nuevas = {cuotas.sum()}')
print(f'30 polígonos: {datos.areas.sum():.2f} ha; adquisición P{p}: {tabla.costo_nuevas_um.sum():.2f} u.m.')
presupuesto_30 = presupuestos(datos,cfg)
print('Diagnóstico económico de los 30 (cuotas fijas; no se modifican):')
display(presupuesto_30[['poligono','presupuesto_um','costo_um','saldo_um',
                      'presupuesto_utilizado_pct','cumple_presupuesto']].round(3))
incumplen = presupuesto_30.loc[~presupuesto_30.cumple_presupuesto,'poligono'].tolist()
print('Incumplen individualmente:',incumplen)
print('Saldo agregado (no compensa automáticamente déficits individuales):',presupuesto_30.saldo_um.sum())


## 3. Conjuntos e índices
Los nodos se identifican por `(polígono, índice local)`; las coordenadas son pares reales $(x,y)$ en metros. El índice local es solo una etiqueta, no una celda binaria de ocupación.

|Símbolo|Significado|Tipo en Python|Unidad|Origen|Equivalente|
|---|---|---|---|---|---|
|$P$|30 polígonos tabulados|lista de int|—|socio|`datos.areas.index`|
|$S$|10 especies|lista de str|—|socio|`datos.S`|
|$F_p$|sitios existentes|array `(nf,2)`|m|simulación, conteos reales|`F`|
|$V_p$|sitios nuevos seleccionados|array `(nv,2)`|m|selección geométrica|`V`|
|$U_p=F_p\cup V_p$|todos los sitios|array `(N,2)`|m|concatenación|`g['U']`|
|$E_p$|pares no dirigidos cercanos|array `(M,2)` de int|—|distancias|`g['pares']`|

En P1 los primeros 165 índices son fijos; los siguientes 677 admiten asignación. Las especies fijas se conocen; **sus coordenadas todavía se simulan**.

In [ ]:
display(pd.DataFrame({'conjunto':['P','S',f'F{p}',f'V{p}',f'U{p}'],
    'cardinalidad':[len(datos.areas),len(S),int(existentes.sum()),int(nuevas.sum()),int(cuotas.sum())]}))

## 4. Parámetros y supuestos
|Símbolo|Significado|Tipo|Unidad|Origen|Python|
|---|---|---|---|---|---|
|$A_p$|área|float|ha|socio|`area`|
|$D$|densidad final objetivo|float|plantas/ha|referencia socio|`cfg.densidad`|
|$T_p$|meta redondeada|int|plantas|regla computacional|`cuotas.sum()`|
|$n_{ps}$|inventario|array int|plantas|socio|`existentes`|
|$R_{ps}$|cuota final|array int|plantas|metas y restos mayores|`cuotas`|
|$q_{ps}=R_{ps}-n_{ps}$|adquisición fija|array int|plantas|derivación|`nuevas`|
|$d_{pij}$|distancia euclidiana|float|m|coordenadas simuladas|`norm(U[i]-U[j])`|
|$d_{min}$|separación ocupada|float|m|supuesto experimental|`cfg.separacion_min`|
|$r$|radio de interacción|float|m|supuesto experimental|`cfg.radio`|
|$C_{st}$|coeficiente competencia|array float|adimensional|supuesto provisional|`C`|
|$w_{pij}$|peso por distancia|array float|adimensional|supuesto|`g['pesos']`|
|$a_{pis}$|especie existente fija|implícita en int|0/1|inventario y permutación|`fijas`|

$T_p=\lfloor A_pD+0.5\rfloor$. Distribuimos $T_p$ según proporciones de las metas por ha: piso y restos mayores, con empates resueltos por orden del catálogo. Si una cuota queda por debajo del inventario, se informa error; no se eliminan plantas.

**P1:** $1.28(658)=842.24$, redondeado a 842. Con 165 existentes hay 677 nuevas. En este recorrido espacial de referencia, $q$ y $R$ son parámetros; en la sección 15 se decide $R'$ mediante el MILP. Se usa $d_{min}=3.4$ m y $r=6$ m: funciones distintas.

### Presupuesto y costo conocidos
$$B_p=12500A_p,\qquad C_p=\sum_s c_sq_{ps},\qquad h_p=B_p-C_p,\qquad u_p=100C_p/B_p.$$

|Símbolo|Significado|Tipo|Unidad|Origen|Python|
|---|---|---|---|---|---|
|$c_s$|costo unitario|float|u.m./planta|socio, diapositiva 18|`datos.especies.costo_unitario`|
|$B_p$|presupuesto disponible|float|u.m.|12500 u.m./ha × área|`presupuesto_um`|
|$C_p$|adquisición|float|u.m.|cuotas fijas e inventario|`costo_um`|
|$h_p$|saldo, negativo = déficit|float|u.m.|diferencia|`saldo_um`|
|$u_p$|porcentaje utilizado|float|%|cálculo|`presupuesto_utilizado_pct`|

P1 dispone de 16000 u.m. y requiere 15903.5: saldo 96.5. La moneda no está especificada. Solo se incluyen plantas; faltan transporte, mantenimiento y riego.


In [ ]:
print('Cuotas:', cuotas.tolist())
print('Compras:', nuevas.tolist())
if p == 1:
    assert (existentes.sum(), nuevas.sum(), cuotas.sum()) == (165,677,842)
else:
    assert np.all(cuotas >= existentes) and cuotas.sum() == existentes.sum() + nuevas.sum()

## 5. Variables de decisión
Para $i\in V_p$:
$$x_{pis}=\begin{cases}1&\text{especie }s\text{ en sitio nuevo }i,\\0&\text{otro caso.}\end{cases}$$
Se define $y_{pis}=a_{pis}$ en $F_p$ y $y_{pis}=x_{pis}$ en $V_p$.

El código almacena una etiqueta entera por planta (`y[i]`), equivalente a una fila one-hot, para ahorrar memoria. Es una representación de **especie asignada**, no un vector de ocupación del terreno. Las coordenadas permanecen en `g['U']`. Por ejemplo, la etiqueta 2 significa la tercera especie del catálogo.

|Símbolo|Significado|Tipo|Unidad|Origen|Python|
|---|---|---|---|---|---|
|$x_{pis}$|Asignación de especie nueva|binario implícito|0/1|decisión heurística|`y[i] == s`, $i\in V_p$|
|$y_{pis}$|Asignación completa|binario implícito|0/1|fija en F, decisión en V|`y[i] == s`|


In [ ]:
ejemplo_etiqueta = 2
print('Especie:', S[ejemplo_etiqueta])
print('Fila binaria equivalente:', np.eye(len(S),dtype=int)[ejemplo_etiqueta])

## 6. Función objetivo
$$\min J=\sum_{p\in P}\sum_{\{i,j\}\in E_p}w_{pij}\sum_{s\in S}\sum_{t\in S} C_{st}y_{pis}y_{pjt},\qquad
w_{pij}=\max(0,1-d_{pij}/r).$$
Cada par no dirigido aparece **una sola vez**. Como cada sitio tiene una especie, el producto one-hot selecciona $C_{s_i,s_j}$. En código: `sum(pesos * C[y[i], y[j]])`.

$J=J_{FF}+J_{FV}+J_{VV}$; $J_{FF}$ es constante al optimizar un escenario. Dos plantas a 4 m con $r=6$ tienen peso $1/3$; si son de la misma especie y $C_{ss}=3$, su contribución es 1. Esto es un índice, **no una probabilidad de supervivencia**.

In [ ]:
display(Code(inspect.getsource(objetivo),language='python'))
print('Ejemplo de una arista:', (1-4/cfg.radio)*3)

## 7. Restricciones
Para cada polígono y especie:
$$\sum_{s\in S}x_{pis}=1\quad(i\in V_p),\qquad
n_{ps}+\sum_{i\in V_p}x_{pis}=R_{ps},$$
$$\sum_s R_{ps}=T_p=|F_p|+|V_p|,\quad y_{pis}=a_{pis}\ (i\in F_p),\quad x_{pis}\in\{0,1\}.$$
Además, $d_{pij}\ge d_{min}$ para todo par ocupado distinto y todos los sitios pertenecen al rectángulo simulado.

**Separación de etapas:** la geometría selecciona exactamente $|V_p|=\sum_s q_{ps}$ sitios compatibles. Después la heurística solo asigna especies; no decide ocupación ni mueve plantas. Permutar etiquetas nuevas conserva las cuotas. Ningún algoritmo garantiza óptimo global o local certificado.

Se corrige la desigualdad $\le1$ del texto antiguo: aquí todos los sitios de $V_p$ deben ocuparse, por lo que corresponde igualdad.

### Factibilidad económica con cuotas fijas
El requisito adicional deseado es $C_p\le B_p$ **para cada polígono**. Como $q$ ya está fijado, ningún intercambio de especies puede reducir C. Si falla, el modelo con cuotas actuales **y presupuesto obligatorio es inviable**, aunque la asignación espacial sea válida. Se mantienen esos experimentos para diagnóstico y se marca `cumple_presupuesto=False`; no se presentan como soluciones que cumplan presupuesto.

La presentación del socio formador (diapositivas 13 y 15) permite variaciones de hasta ±10% por especie manteniendo el total. La interpretación diagnóstica usada es:
$$\max(n_{ps},\lceil0.9R_{ps}\rceil)\le R'_{ps}\le\lfloor1.1R_{ps}\rfloor,\quad \sum_sR'_{ps}=T_p,\quad q'_{ps}=R'_{ps}-n_{ps}.$$
Minimizar $\sum_sc_sq'_{ps}$ en esa banda es un problema económico auxiliar: partir de límites inferiores y completar por costo creciente da el mínimo (cada intercambio hacia una especie más barata reduce costo). Se conserva densidad total, integralidad e inventario. **No altera R ni q del experimento**. La referencia de la banda sobre R redondeada y los requisitos ecológicos deben confirmarse con el socio.

Si ese mínimo supera B, ni siquiera esa flexibilidad basta. Si es menor, solo demuestra posibilidad aritmética: no certifica equivalencia ecológica. La sección 15 decide R' antes de DSATUR, impone B y revalida las cuotas; siguen pendientes requisitos ecológicos medidos. Ni el precio ni J provisional demuestran supervivencia.


In [ ]:
assert np.all(cuotas >= existentes)
assert nuevas.sum() + existentes.sum() == cuotas.sum()
print('Cuotas no negativas y meta conciliada; validaciones espaciales después de construir sitios.')
flexibilidad_10 = diagnostico_flexibilidad(datos,cfg)
display(flexibilidad_10.loc[flexibilidad_10.poligono.isin(incumplen),
    ['poligono','costo_fijo_um','presupuesto_um','costo_minimo_banda_10_um',
     'presupuesto_alcanzable_en_banda','aplicado_al_modelo']])


## 8. Construcción geométrica de posiciones
El rectángulo tiene superficie $10^4A_p$ m²; aspecto 1 por defecto. Muestreamos posiciones existentes con rechazo por separación y permutamos las especies conservando los conteos. Un límite de propuestas impide ciclos sin fin: agotarlo no prueba imposibilidad matemática.

**Figura 1:** cada símbolo representa una especie en las posiciones simuladas existentes. No son coordenadas observadas. Después generamos una malla triangular de paso 3.4 m, descartamos sitios cercanos a existentes y elegimos por dispersión la cantidad requerida de posiciones nuevas (677 en P1). Este criterio no utiliza C ni especies. Una malla insuficiente produce un fallo registrado, sin relajar distancias.

La densidad ideal de una malla triangular a 3.4 m es diferente de 658 plantas/ha: solo ocupamos parte de la malla. La selección y DSATUR pueden crecer aproximadamente de forma cuadrática; no se guardan matrices completas de distancias.

### Qué es aleatorio y qué es determinista

|Etapa|Comportamiento|Qué permanece fijo|
|---|---|---|
|Inventarios y cuotas|Deterministas, provienen de datos y regla de redondeo|n, R, q y costo|
|Existentes|Propuestas uniformes en el rectángulo; rechazo si violan separación|Conteos y distancia mínima|
|Especies existentes en escenario|Permutación aleatoria de etiquetas de conteos conocidos|Inventario exacto por especie|
|Candidatas|Malla triangular determinista, filtrada usando F simulado|Paso y regla geométrica|
|Sitios nuevos|Elección sucesiva del más alejado de los sitios ocupados|Cantidad exacta de nuevos|
|DSATUR|Determinista dado grafo, C y cuotas; desempates fijos|Posiciones y existentes|
|Referencia y mejora local|Permutación e intercambios con flujos aleatorios separados|Cuotas y geometría del escenario|

Las posiciones nuevas **no son puntos uniformes independientes**. Se determinan a partir de una malla y de las existentes simuladas; por eso varían entre escenarios aunque su selector sea determinista. El rechazo también hace dependientes las posiciones existentes. Durante la optimización, las plantas existentes quedan inmutables; que cambien entre escenarios no significa que el algoritmo las mueva en una solución.

Las coordenadas (0,0) son un origen local artificial. Cada polígono tiene su propio rectángulo; no conocemos orientación, ubicación real ni un sistema de referencia geográfico común. No se deben unir en un mapa global ficticio.


In [ ]:
limites = dimensiones(area,cfg)
rng_pos, rng_base, rng_local = flujos(p,0,cfg)
F,fijas = simular_existentes(existentes,limites,cfg,rng_pos)
mostrar(vis.mapa(F,fijas,S,limites,f'P{p}: {len(F)} existentes; coordenadas simuladas'),'01_existentes')
pool,V = candidatas(F,limites,int(nuevas.sum()),cfg)
print(f'{len(pool)} candidatas admisibles; {len(V)} seleccionadas; {len(F)+len(V)} sitios ocupados.')

**Interpretación:** se conservan las etiquetas existentes y se selecciona exactamente el número requerido de sitios nuevos (165 existentes y 677 nuevos en el ejemplo P1). Los espacios vacíos no son decisiones binarias del optimizador de especies.

## 9. Grafo espacial y matriz intercambiable
Conectamos sitios separados a lo más 6 m y guardamos un peso por arista. No hay interacciones entre polígonos.

`config/competencia.json` contiene el orden de especies y una matriz explícita: 3 en la diagonal, 2 entre especies distintas del mismo género y 1 entre géneros diferentes. El cargador valida dimensión, simetría, finitud, no negatividad y orden del catálogo. **Son supuestos**, no mediciones ecológicas.

Más adelante se podría construir C con consumos normalizados de agua y nutrientes, pesos justificados y calibración con datos de campo. Bastaría reemplazar el JSON conservando las propiedades requeridas; no habría que modificar DSATUR ni los deltas.

In [ ]:
C = cargar_competencia(ROOT/'config/competencia.json',S)
g = construir_grafo(p,F,V,cfg)
display(pd.DataFrame(C,index=S,columns=S))
print(f"Nodos: {len(g['U'])}; aristas: {len(g['pares'])}")

## 10. Referencia aleatoria y DSATUR adaptado
La referencia permuta el multiconjunto de especies nuevas: es factible y usa idénticos sitios y cuotas. DSATUR elige el siguiente vértice por saturación (número de especies distintas entre vecinos asignados), luego grado ponderado y menor índice.

Para cada especie disponible calcula $\Delta_i(s)=\sum_{j\in N(i),j\ asignado}w_{ij}C_{s,s_j}$; escoge menor costo incremental, mayor cuota pendiente y menor índice de especie, en ese orden. **No es coloreo propio:** repetir especies se permite con penalización. Como todas las especies pueden ocupar cualquier sitio nuevo y las cuotas suman vacantes, siempre hay especie disponible.

La implementación extraída se muestra para poder relacionar saturación, costos y cuotas con el código. Los desempates y el orden de aristas se conservan del prototipo.

In [ ]:
display(Code(inspect.getsource(dsatur),language='python'))
t0=perf_counter();y_base=base_aleatoria(fijas,nuevas,rng_base);t_base=perf_counter()-t0
t0=perf_counter();y_dsatur=dsatur(fijas,nuevas,g,C);t_dsatur=perf_counter()-t0
print(f'Aleatoria: {objetivo(y_base,g,C):.3f}; DSATUR: {objetivo(y_dsatur,g,C):.3f}')

## 11. DSATUR + búsqueda local
1. Proponer dos sitios **nuevos distintos** uniformemente al azar.
2. Si tienen la misma especie, contar y descartar la propuesta.
3. Evaluar $\Delta J$ sumando aristas incidentes. La arista entre ambos sitios no cambia porque C es simétrica.
4. Intercambiar solo si hay mejora negativa. Se conserva la tolerancia histórica: $\Delta J<-10^{-10}$, para descartar empates de coma flotante.

Nunca se modifican plantas existentes. Cuotas y coordenadas quedan fijas. El presupuesto cuenta propuestas, incluidas las descartadas; terminarlo no certifica óptimo local.

El diagnóstico opcional de trayectoria tiene **propuestas en X y J en Y**. Descensos = mejoras aceptadas; mesetas = sin mejora aceptada. Incluye la meseta hasta la última propuesta. Se muestran contadores que satisfacen propuestas = iguales + evaluados y evaluados = aceptados + rechazados.

In [ ]:
display(Code(inspect.getsource(busqueda_local),language='python'))
y_final,historia,stats = busqueda_local(y_dsatur,g,C,rng_local,cfg.intentos(len(V)))
display(pd.Series(stats,name='Búsqueda local'))
detalle = dict(g=g,F=F,V=V,fijas=fijas,limites=limites,historia=historia)
if cfg.diagnosticos:
    for nombre,fig in vis.diagnosticos(detalle,C,S):
        mostrar(fig,'diagnostico_'+nombre)

**Interpretación:** el número de aceptados es menor que el de evaluados; el índice final puede disminuir sin modificar las compras. Los cinco diagnósticos opcionales se activan con `diagnosticos=True` en la configuración.

## 12. Validación de las soluciones
La siguiente celda comprueba cuotas, plantas existentes, meta total, pertenencia al terreno y separación mínima para los tres métodos. También calcula J directamente desde las aristas, comprueba los contadores de búsqueda local y verifica que la mejora no aumente J.

Estas comprobaciones también forman parte de `ejecutar_escenario`. Las baterías `verificar_modelo.py` y `verificar_economia.py` comprueban regresión, economía, integración y errores controlados.

In [ ]:
for y in (y_base,y_dsatur,y_final):
    validar_solucion(y,g,fijas,existentes,cuotas,limites,cfg)
assert np.array_equal(g['U'][:len(F)],F)
j_directo = sum(a['weight']*C[y_final[i[1]],y_final[j[1]]] for i,j,a in g['G'].edges(data=True))
assert np.isclose(j_directo,objetivo(y_final,g,C))
assert stats['propuestas'] == stats['especies_iguales']+stats['evaluados']
assert stats['evaluados'] == stats['aceptados']+stats['rechazados']
assert objetivo(y_final,g,C) <= objetivo(y_dsatur,g,C)+1e-8
print('Las tres soluciones cumplen restricciones; evaluación directa e incremental consistentes.')

## 13. Resultados principales: aleatoria frente a DSATUR + búsqueda local
**Figura 2:** distribución final, con anillos para existentes. **Figura 3:** comparación de dos procedimientos sobre idénticos sitios y cuotas. El valor DSATUR intermedio permanece en `comparacion.csv`, pero no es una tercera alternativa en las gráficas principales.

La reducción `100 × (J_base − J_final)/J_base` corresponde al procedimiento completo. El descenso entre J_dsatur y J_final mide solo la etapa de intercambios y puede verse en el diagnóstico opcional de búsqueda local. Ninguna reducción representa un aumento medido de supervivencia.


In [ ]:
mostrar(vis.mapa(g['U'],y_final,S,limites,f'P{p}: distribución final optimizada',nf=len(F)),'02_final')
comparacion = pd.DataFrame([dict(metodo=nombre,J=objetivo(y,g,C),**objetivo(y,g,C,True),segundos_asignacion=tiempo)
    for nombre,y,tiempo in [('Aleatoria',y_base,t_base),('DSATUR',y_dsatur,t_dsatur),('DSATUR + local',y_final,t_dsatur+stats['segundos'])]])
display(comparacion.loc[comparacion.metodo.ne('DSATUR')].round(4))
mostrar(vis.comparacion(comparacion,f'P{p}, escenario 0: mismos sitios y cuotas'),'03_comparacion')
economia_p = presupuesto_30.set_index('poligono').loc[p]
for clave in ('presupuesto_um','costo_um','saldo_um','presupuesto_utilizado_pct','cumple_presupuesto'):
    comparacion[clave] = economia_p[clave]
comparacion['escenario'] = 0
comparacion.to_csv(OUT/'comparacion.csv',index=False)
pd.DataFrame({'indice':np.arange(len(y_final)),'x_m':g['U'][:,0],'y_m':g['U'][:,1],
    'existente':np.arange(len(y_final))<len(F),'especie':[S[s] for s in y_final]}).to_csv(OUT/'distribucion.csv',index=False)
print(f'Reducción frente a aleatoria: {100*(1-objetivo(y_final,g,C)/objetivo(y_base,g,C)):.2f}%')

**Referencia histórica de P1:** la referencia esperada es 1089.007, DSATUR 991.748 y DSATUR + local 874.525 con las semillas y versiones fijadas. La tabla anterior contiene los valores efectivamente ejecutados y separa el término fijo FF.

## 14. Monte Carlo espacial
Cada corrida es un **escenario espacial completo** del polígono detallado, con los mismos conteos de inventario y nuevas plantas. Si se configura P1 con 12 escenarios, son 12 realizaciones con 165 existentes y 677 nuevas cada uno. No son doce posiciones individuales. Se regeneran posiciones y se aplican nuevamente los métodos; cada resultado se adapta a su escenario.

Los conteos y compras siguen fijos. La variación mezcla geometría, referencia aleatoria y propuestas locales: no modela incertidumbre sobre cuánto comprar ni valida ecología. DSATUR es determinista dado el grafo. Los flujos dependen de `(semilla,p,escenario)`, por lo que cambiar el orden del lote no cambia soluciones.

**Figura 4:** comparación pareada y distribución empírica de reducciones. Fallos no se sustituyen ni se ocultan; las estadísticas se condicionan a los completados. Percentiles son descriptivos, no intervalos de confianza. Los tiempos dependen del equipo.

### Escenario espacial y semillas
Un escenario es la realización completa de un polígono: todas las posiciones existentes, malla filtrada, sitios nuevos y asignaciones. La semilla base y `(polígono, escenario)` alimentan `SeedSequence(...).spawn(3)` para separar posiciones, referencia y búsqueda local. Cambiar el orden de ejecución no cambia los resultados; mantener solo la semilla sin fijar versiones tampoco garantiza identidad entre cualquier versión de librería.

En el experimento global, el escenario e reúne una realización de cada uno de los 30 polígonos; sus flujos son independientes por ID. Se suma J dentro de ese conjunto, nunca entre escenarios. El algoritmo observa cada geometría y se adapta a ella: no se evalúa una única decisión robusta previa.

La variación mezcla geometría y aleatoriedad de la referencia y búsqueda local. No se modelan inventarios desconocidos, mortalidad futura, disponibilidad de agua, suelo, topografía real ni incertidumbre en compras o precios. Los porcentajes históricos de supervivencia permanecen como contexto, sin convertirse en predicciones.


In [ ]:
cfg_mc = replace(cfg,poligonos=(p,))
mc = ejecutar_lote(datos,cfg_mc,C)
guardar_monte_carlo_por_poligono(RESULTADOS,mc)
display(mc.reindex(columns=['poligono','escenario','estado','detalle','J_base','J_final',
                          'mejora_pct','cumple_presupuesto']).round(4))
ok = mc.loc[mc.estado.eq('ok')]
print(f'Completados {len(ok)}/{len(mc)}; fallos u omisiones: {len(mc)-len(ok)}')
if not ok.empty:
    resumen = ok[['J_base','J_final','mejora_pct']].agg(['count','mean','std','min','max']).T
    display(resumen.round(4))
mostrar(vis.monte_carlo(mc,f'P{p}: escenarios espaciales completos'),'04_monte_carlo')


**Interpretación:** el denominador de escenarios completados forma parte del resultado. Con la configuración original P1, el historial registra 12/12 escenarios y una reducción media del 20.15%; la tabla anterior permite repetir esa comprobación. No extrapolar el índice a supervivencia. Si reduces `ESCENARIOS_MC`, las estadísticas se recalcularán con menos escenarios.

### Experimento opcional: presupuesto de búsqueda local
La celda siguiente está desactivada por defecto. Al establecer `EJECUTAR_COMPARACION_PRESUPUESTOS = True` en la primera celda, compara sobre tres escenarios las mismas semillas con 6000 propuestas fijas frente a `10 ×` nuevas propuestas. En P1 equivale a 6770 propuestas proporcionales. No se garantiza que el mayor presupuesto mejore en todos los casos.


In [ ]:
if EJECUTAR_COMPARACION_PRESUPUESTOS:
    presupuestos = []
    for modo in ('fijo', 'proporcional'):
        config_presupuesto = replace(cfg, poligonos=(p,), escenarios=3, presupuesto=modo)
        ensayo = ejecutar_lote(datos, config_presupuesto, C)
        guardar_ejecucion(OUT/('presupuesto_'+modo), datos, config_presupuesto, C, ensayo)
        ensayo['presupuesto'] = modo
        presupuestos.append(ensayo)
    presupuestos = pd.concat(presupuestos, ignore_index=True)
    display(presupuestos[['escenario','presupuesto','estado','local_propuestas',
                          'local_aceptados','J_final','seg_local']].round(4))
else:
    print('Comparación de presupuestos omitida (opcional).')


## 15. Etapa económica e integración con la etapa espacial

Para cada polígono, $R_s$ es la cuota final original, $n_s$ el inventario, $c_s$ el precio y $B=12500A$ el presupuesto. Las decisiones enteras son $R'_s$ y $q_s=R'_s-n_s$.

$$\sum_s R'_s=\sum_s R_s,\qquad \max(n_s,\lceil0.9R_s\rceil)\le R'_s\le\lfloor1.1R_s\rfloor,\qquad \sum_sc_s(R'_s-n_s)\le B.$$

La banda se interpreta sobre **cuotas finales originales**, pendiente de confirmación del socio. No hay transferencias entre polígonos.

### Objetivo y método

$$\min D=\sum_{s:R_s>0}\frac{d_s}{R_s},\qquad d_s\ge R'_s-R_s,\quad d_s\ge R_s-R'_s,\quad d_s\ge0.$$

La desviación absoluta trata cada planta igual, pero puede concentrar cambios proporcionalmente grandes en especies raras. Elegimos desviaciones relativas: cambiar una planta de una especie poco frecuente pesa más. La banda protege además cada proporción. Una combinación exigiría elegir un peso ecológico adicional sin evidencia; por eso no se introduce. Si $R_s=0$, la banda obliga $R'_s=0$ y se omite su término, sin dividir por cero.

`scipy.optimize.milp` resuelve primero D y después el costo fijando el valor de D. La escala común de los pesos no cambia el objetivo. El segundo resultado solo se acepta si su desviación racional es **exactamente igual** y no aumenta el costo. Los certificados se muestran por separado; un límite computacional no certifica óptimo ni inviabilidad. Los conteos y el presupuesto se revalidan con aritmética entera/decimal, sin ampliar B.

### Integración y comparaciones

`DatosConCuotas` entrega $R'$ y $q$ a los módulos espaciales originales. Se conservan total, inventario, posiciones, distancias, C, J y búsqueda local. Como no hay prohibiciones especie-sitio, cualquier cuota no negativa que sume el total cabe en una geometría ya construida. Un fallo de construcción se reporta y no demuestra imposibilidad en todo el terreno.

La siguiente celda ejecuta ambos modos bajo **las mismas semillas, parámetros y versiones**. Para cuotas sin cambios reutiliza la referencia fija de esa misma ejecución; para cuotas distintas verifica igualdad exacta de coordenadas y existentes. Cada fila compara el mismo polígono y escenario. Un cambio positivo en J significa mayor competencia.

Se guardan `planes.json`, `cuotas.csv`, `economia.csv`, `escenarios.csv`, `comparacion_pareada.csv`, manifestación de versiones/hashes y figuras. Los tiempos económicos se cuentan una sola vez por polígono. Las carpetas históricas `resultados/P*/` y `resultados/modelo_completo/` no se sobrescriben.

In [ ]:
from reforestacion.dos_etapas import ejecutar
ruta_integrada, economia, lote_integrado, pares = ejecutar(
    datos, cfg_lote, C, modo=MODO, comparar=True, graficas=True,
    progreso=lambda f: print(f"P{f['poligono']}, escenario {f['escenario']}: {f['estado_factibilidad']}"))
display(economia[['poligono','presupuesto_um','costo_original_um','costo_ajustado_um',
                  'ahorro_um','certificado_desviacion','certificado_desempate']].round(3))
display(lote_integrado.reindex(columns=['poligono','escenario','estado_factibilidad',
                                       'J_base','J_dsatur','J_final','seg_espacial_adicional']))
if not pares.empty:
    display(pares.groupby('poligono')[['cambio_J_final','cambio_J_final_pct']].mean().round(4))
from IPython.display import Image
for archivo in (ruta_integrada/'economia.png', ruta_integrada/'cambio_competencia.png'):
    if archivo.exists(): display(Image(filename=str(archivo)))
print('Resultados de esta ejecución:',ruta_integrada)

## 16. Interpretación y límites

El modo fijo reproduce las cuotas originales aunque excedan el presupuesto. El económico impone el presupuesto antes de distribuir especies. Solo `factible_economica_y_espacialmente` confirma ambas validaciones. Se distinguen inviabilidad económica, geometría no construida, errores y omisiones por recursos; los índices faltantes no son ceros.

El mínimo económico auxiliar permite informar cuánto presupuesto se necesita si la banda no basta. El objetivo principal preserva composición, por lo que no necesariamente alcanza ese costo mínimo. La optimalidad económica se refiere al MILP y sus tolerancias numéricas; la etapa espacial sigue siendo heurística.

La comparación pareada mide el efecto de cambiar cuotas bajo el mismo escenario. No se garantiza un J menor al cambiar composición; sí se verifica que la búsqueda local no aumente J respecto a DSATUR dentro de cada modo.

Siguen pendientes confirmar la interpretación del ±10%, geometría real, C calibrada, P13, disponibilidad de plantas, costos integrales y requisitos ecológicos. Monte Carlo simula posiciones y asignaciones, no incertidumbre de compras ni supervivencia.

### Ejecutar las etapas por separado

Economía únicamente: `python scripts/ejecutar_dos_etapas.py --etapa economica --poligonos todos`.

Para usar un plan ya guardado, carga `planes.json`, convierte las claves a enteros, construye `DatosConCuotas(datos, planes)` y llama a `ejecutar_escenario` o `ejecutar_lote` con esa vista. Se revalidan bandas y presupuesto contra los datos actuales. Ver ejemplo completo en README.

In [ ]:
print('Modo:', MODO)
print('Estados:', lote_integrado.estado_factibilidad.value_counts().to_dict())
print('Ahorro total de adquisición:', economia.ahorro_um.sum(), 'u.m.')
print('Tiempo económico total:', economia.seg_economia.sum(), 's')
if not pares.empty:
    print('Comparaciones pareadas:', len(pares))
    totales = pares.groupby('escenario')[['J_final_fijo','J_final_economico','cambio_J_final']].sum()
    completos = pares.groupby('escenario').poligono.nunique().eq(len(poligonos_ids))
    totales.loc[~completos, :] = np.nan
    totales['completo'] = completos
    display(totales)